# 🤖 AI Tutor by Collins Aerospace
### Full AI Tutor with Vision RAG (Groq Powered)

## 1️⃣ Install Necessary Libraries

In [ ]:
!pip install groq gradio pytesseract pillow sentence-transformers faiss-cpu

## 2️⃣ Import Libraries

In [ ]:
from groq import Groq
import gradio as gr
from PIL import Image
import pytesseract
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

## 3️⃣ Configure Groq API

In [ ]:
client = Groq(api_key="YOUR_GROQ_API_KEY")

## 4️⃣ Core AI Logic

In [ ]:
def call_llm(system_prompt, user_prompt):
    completion = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return completion.choices[0].message.content

## 5️⃣ LLM Decision Engine

In [ ]:
def decide_task(user_input):
    decision_prompt = f"""
    Classify into: chat, lesson, quiz, project.
    Input: {user_input}
    Return only category.
    """

    category = call_llm("You are a classifier.", decision_prompt).lower()

    if "lesson" in category:
        return call_llm("Expert teacher", f"Create lesson plan: {user_input}")

    elif "quiz" in category:
        return call_llm("Quiz generator", f"Generate 5 MCQs with answers: {user_input}")

    elif "project" in category:
        return call_llm("Project mentor", f"Suggest project ideas: {user_input}")

    else:
        return call_llm("Helpful tutor", user_input)

## 6️⃣ Vision RAG Setup

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
dimension = 384
index = faiss.IndexFlatL2(dimension)
documents = []

In [ ]:
def extract_text_from_image(image):
    return pytesseract.image_to_string(image)

In [ ]:
def add_to_knowledge_base(text):
    emb = embedder.encode([text])
    index.add(np.array(emb).astype('float32'))
    documents.append(text)

In [ ]:
def retrieve_context(query, k=2):
    emb = embedder.encode([query])
    D, I = index.search(np.array(emb).astype('float32'), k)

    results = []
    for i in I[0]:
        if i < len(documents):
            results.append(documents[i])

    return "\n".join(results)

In [ ]:
def evaluate_with_vision_rag(image):
    if image is None:
        return "Upload an image"

    text = extract_text_from_image(image)
    add_to_knowledge_base(text)
    context = retrieve_context(text)

    prompt = f"""
    Evaluate student answer.

    Answer:
    {text}

    Context:
    {context}

    Give:
    - Score /10
    - Feedback
    - Improvements
    - Correct Answer
    """

    return call_llm("Strict evaluator", prompt)

## 7️⃣ Professional UI (Gradio)

In [ ]:
custom_css = """
body { background-color: #f7f9fc; }
.header { text-align:center; font-size:28px; font-weight:bold; }
.subheader { text-align:center; color:gray; }
.line {
 height:4px;
 background:linear-gradient(90deg,#3b82f6,#06b6d4);
 animation:move 2s linear infinite;
}
@keyframes move {
 0%{background-position:0%}
 100%{background-position:100%}
}
"""

def ai_tutor_ui(user_input):
    if not user_input:
        return "Enter a query"
    return decide_task(user_input)

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    gr.HTML("""
    <div class='header'>AI Tutor</div>
    <div class='subheader'>by Collins Aerospace</div>
    <div class='line'></div>
    """)

    with gr.Tab("💬 Tutor"):
        inp = gr.Textbox(label="Ask anything")
        out = gr.Textbox(lines=12)
        btn = gr.Button("Generate")
        btn.click(ai_tutor_ui, inp, out)

    with gr.Tab("🖼️ Vision Evaluation"):
        img = gr.Image(type="pil")
        eval_out = gr.Textbox(lines=15)
        eval_btn = gr.Button("Evaluate")
        eval_btn.click(evaluate_with_vision_rag, img, eval_out)

demo.launch()